# 微調 Open AI 模型

本筆記本基於 Open AI 提供的 [微調](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst) 文件中的最新指引。

> **注意：** 本筆記本的範例輸出是使用 `gpt-3.5-turbo` 生成的，該模型現已在 Microsoft Foundry 的 Azure OpenAI 服務中退休，不再支援推理和微調（OpenAI 也已直接淘汰該模型）。以下的操作步驟和概念仍然準確，但如果您今天開始新微調工作，建議選擇目前支援的模型，例如 `gpt-4o-mini` 或 `gpt-4.1-mini`。詳情請參考[微調模型列表](https://learn.microsoft.com/en-us/azure/ai-foundry/foundry-models/concepts/models-sold-directly-by-azure?WT.mc_id=academic-105485-koreyst#fine-tuning-models)以獲得當前可微調的模型。

微調通過使用與特定用例或場景相關的額外數據和上下文重新訓練基礎模型，以提升其在您的應用上的表現。請注意，類似 _few shot learning_ 和 _retrieval augmented generation_ 的提示工程技術可以通過相關數據增強預設提示以提升品質，但這些方法受到所用基礎模型最大 token 窗口大小的限制。

使用微調，我們實際上是在用所需資料重新訓練模型本身（允許我們使用比最大 token 窗口能容納的更多範例）——並部署一個 _自訂_ 版本的模型，這樣推理時就不需要提供範例。這不僅提升了提示設計的效能（讓我們在 token 窗口使用上更有彈性），也可能降低成本（減少推理時需送給模型的 token 數量）。

微調有四個步驟：
1. 準備訓練資料並上傳。
1. 執行訓練工作以取得微調模型。
1. 評估微調模型並反覆優化品質。
1. 滿意後部署微調模型以供推理使用。

請注意並非所有基礎模型都支援微調——請參閱[OpenAI 文件](https://platform.openai.com/docs/guides/fine-tuning/what-models-can-be-fine-tuned?WT.mc_id=academic-105485-koreyst)獲取最新資訊。您也可以微調先前已微調的模型。本教程中，我們將使用 `gpt-35-turbo` 作為微調的目標基礎模型（有關最新、支援模型的說明，請見上方備註）。

---


### 步驟 1.1：準備您的資料集

讓我們打造一個聊天機器人，幫助您透過以一首打油詩回答關於元素的問題，進而了解元素週期表。在_這個_簡單的教學中，我們將只製作一個資料集，用一些範例回應來訓練模型，這些範例展示了資料的預期格式。在真實世界的使用案例中，您需要建立一個包含更多範例的資料集。如果已存在適用於您的應用領域的開放資料集，您也可以重新格式化它來用於微調。

由於我們專注於 `gpt-35-turbo` 且尋求單輪回應（聊天完成），我們可以使用[這個建議格式](https://platform.openai.com/docs/guides/fine-tuning/preparing-your-dataset?WT.mc_id=academic-105485-koreyst)來建立範例，以符合OpenAI聊天完成的要求。如果您預期多輪的對話內容，則會使用包含 `weight` 參數的[多輪範例格式](https://platform.openai.com/docs/guides/fine-tuning/multi-turn-chat-examples?WT.mc_id=academic-105485-koreyst)，以表示哪些訊息應該在微調過程中使用（或不使用）。

在本教學中，我們將使用較簡單的單輪格式。資料格式為[jsonl格式](https://jsonlines.org/?WT.mc_id=academic-105485-koreyst)，每行一條記錄，每條為JSON格式的物件。以下程式碼片段展示了2條記錄的範例 — 請參考[training-data.jsonl](../../../../../18-fine-tuning/python/openai/training-data.jsonl)獲取完整樣本集（10個範例），我們會用於本微調教學。<strong>注意：</strong>每條記錄_必須_定義於單獨一行（不能像格式化JSON檔般斷行）。

```json
{ "messages": [{"role": "system", "content": "Elle is a factual chatbot that answers questions about elements in the periodic table with a limerick"}, {"role": "user", "content": "Tell me about Gallium"}, {"role": "assistant", "content": "Gallium, oh gallium, so light - Melts in your hand, oh what a sight - At 86 degrees - Its liquid with ease - And in semiconductors, it's out of sight"}]}
{ "messages": [{"role": "system", "content": "Elle is a factual chatbot that answers questions about elements in the periodic table with a limerick"}, {"role": "user", "content": "Tell me about Hydrogen"}, {"role": "assistant", "content": "Hydrogen, the first in the line - The lightest of all, so divine - It's in water, you see - And in stars, it's the key - The universe's most common sign"}]}
```

在實務中，您會需要一個更大的範例集才能獲得良好結果 — 取捨點在於回應品質與微調所需時間／成本。我們這裡使用較小的範例集，能讓微調快速完成以展示流程。請參考[這個OpenAI Cookbook範例](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_finetune_chat_models.ipynb?WT.mc_id=academic-105485-koreyst)來進一步學習更複雜的微調教學。


---

### 第1.2步 上傳您的資料集

使用 Files API 上傳資料，[如此處所述](https://platform.openai.com/docs/guides/fine-tuning/upload-a-training-file)。請注意，要執行此程式碼，您必須先完成以下步驟：
 - 安裝 `openai` Python 套件（確保使用版本 >=0.28.0 以獲得最新功能）
 - 設置環境變數 `OPENAI_API_KEY` 為您的 OpenAI API 密鑰
欲了解更多資訊，請參閱本課程提供的[設置指南](./../../../00-course-setup/02-setup-local.md?WT.mc_id=academic-105485-koreyst)。

現在，執行程式碼，從您的本地 JSONL 檔案建立一個用於上傳的檔案。


In [24]:
from openai import OpenAI
client = OpenAI()

ft_file = client.files.create(
  file=open("./training-data.jsonl", "rb"),
  purpose="fine-tune"
)

print(ft_file)
print("Training File ID: " + ft_file.id)

FileObject(id='file-JdAJcagdOTG6ACNlFWzuzmyV', bytes=4021, created_at=1715566183, filename='training-data.jsonl', object='file', purpose='fine-tune', status='processed', status_details=None)
Training File ID: file-JdAJcagdOTG6ACNlFWzuzmyV


---

### 步驟 2.1：使用 SDK 建立微調工作


In [25]:
from openai import OpenAI
client = OpenAI()

ft_filejob = client.fine_tuning.jobs.create(
  training_file=ft_file.id, 
  model="gpt-3.5-turbo"
)

print(ft_filejob)
print("Fine-tuning Job ID: " + ft_filejob.id)

FineTuningJob(id='ftjob-Usfb9RjasncaZ5Cjbuh1XSCh', created_at=1715566184, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(n_epochs='auto', batch_size='auto', learning_rate_multiplier='auto'), model='gpt-3.5-turbo-0125', object='fine_tuning.job', organization_id='org-EZ6ag0n0S6Zm8eV9BSWKmE6l', result_files=[], seed=830529052, status='validating_files', trained_tokens=None, training_file='file-JdAJcagdOTG6ACNlFWzuzmyV', validation_file=None, estimated_finish=None, integrations=[], user_provided_suffix=None)
Fine-tuning Job ID: ftjob-Usfb9RjasncaZ5Cjbuh1XSCh


---

### 步驟 2.2：檢查工作的狀態

以下是您可以使用 `client.fine_tuning.jobs` API 執行的一些操作：
- `client.fine_tuning.jobs.list(limit=<n>)` - 列出最近的 n 個微調工作
- `client.fine_tuning.jobs.retrieve(<job_id>)` - 獲取特定微調工作的詳細資訊
- `client.fine_tuning.jobs.cancel(<job_id>)` - 取消微調工作
- `client.fine_tuning.jobs.list_events(fine_tuning_job_id=<job_id>, limit=<b>)` - 列出該工作最多 n 筆事件
- `client.fine_tuning.jobs.create(model="gpt-35-turbo", training_file="your-training-file.jsonl", ...)`

流程的第一步是 _驗證訓練檔案_，確保資料格式正確。


In [26]:
from openai import OpenAI
client = OpenAI()

# List 10 fine-tuning jobs
client.fine_tuning.jobs.list(limit=10)

# Retrieve the state of a fine-tune
client.fine_tuning.jobs.retrieve(ft_filejob.id)

# List up to 10 events from a fine-tuning job
client.fine_tuning.jobs.list_events(fine_tuning_job_id=ft_filejob.id, limit=10)

SyncCursorPage[FineTuningJobEvent](data=[FineTuningJobEvent(id='ftevent-GkWiDgZmOsuv4q5cSTEGscY6', created_at=1715566184, level='info', message='Validating training file: file-JdAJcagdOTG6ACNlFWzuzmyV', object='fine_tuning.job.event', data={}, type='message'), FineTuningJobEvent(id='ftevent-3899xdVTO3LN7Q7LkKLMJUnb', created_at=1715566184, level='info', message='Created fine-tuning job: ftjob-Usfb9RjasncaZ5Cjbuh1XSCh', object='fine_tuning.job.event', data={}, type='message')], object='list', has_more=False)

In [30]:
# Once the training data is validated
# Track the job status to see if it is running and when it is complete
from openai import OpenAI
client = OpenAI()

response = client.fine_tuning.jobs.retrieve(ft_filejob.id)

print("Job ID:", response.id)
print("Status:", response.status)
print("Trained Tokens:", response.trained_tokens)

Job ID: ftjob-Usfb9RjasncaZ5Cjbuh1XSCh
Status: running
Trained Tokens: None


---

### 步驟 2.3：追蹤事件以監控進度


In [44]:
# You can also track progress in a more granular way by checking for events
# Refresh this code till you get the `The job has successfully completed` message
response = client.fine_tuning.jobs.list_events(ft_filejob.id)

events = response.data
events.reverse()

for event in events:
    print(event.message)

Step 85/100: training loss=0.14
Step 86/100: training loss=0.00
Step 87/100: training loss=0.00
Step 88/100: training loss=0.07
Step 89/100: training loss=0.00
Step 90/100: training loss=0.00
Step 91/100: training loss=0.00
Step 92/100: training loss=0.00
Step 93/100: training loss=0.00
Step 94/100: training loss=0.00
Step 95/100: training loss=0.08
Step 96/100: training loss=0.05
Step 97/100: training loss=0.00
Step 98/100: training loss=0.00
Step 99/100: training loss=0.00
Step 100/100: training loss=0.00
Checkpoint created at step 80 with Snapshot ID: ft:gpt-3.5-turbo-0125:bitnbot::9OFWyyF2:ckpt-step-80
Checkpoint created at step 90 with Snapshot ID: ft:gpt-3.5-turbo-0125:bitnbot::9OFWyzhK:ckpt-step-90
New fine-tuned model created: ft:gpt-3.5-turbo-0125:bitnbot::9OFWzNjz
The job has successfully completed


### 第 2.4 步：在 OpenAI 儀表板查看狀態


您也可以透過造訪 OpenAI 網站並瀏覽平台的 _Fine-tuning_ 區段來查看狀態。這將顯示當前工作的狀態，並讓您追蹤之前工作執行的歷史。在這張截圖中，您可以看到先前的執行失敗，第二次執行成功。作為背景說明，這發生在第一次執行使用了格式錯誤的 JSON 檔案記錄 - 修正後，第二次執行成功完成並使模型可供使用。

![Fine-tuning job status](../../../../../translated_images/zh-TW/fine-tuned-model-status.563271727bf7bfba.webp)


您也可以在視覺儀表板中向下捲動，查看狀態訊息和指標，如下所示：

| Messages | Metrics |
|:---|:---|
| ![Messages](../../../../../translated_images/zh-TW/fine-tuned-messages-panel.4ed0c2da5ea1313b.webp) |  ![Metrics](../../../../../translated_images/zh-TW/fine-tuned-metrics-panel.700d7e4995a65229.webp)|


---

### 步驟 3.1：取得 ID 並在程式碼中測試微調模型


In [46]:
# Retrieve the identity of the fine-tuned model once ready
response = client.fine_tuning.jobs.retrieve(ft_filejob.id)
fine_tuned_model_id = response.fine_tuned_model
print("Fine-tuned Model ID:", fine_tuned_model_id)

Fine-tuned Model ID: ft:gpt-3.5-turbo-0125:bitnbot::9OFWzNjz


In [ ]:
# You can then use that model to generate completions from the SDK as shown
# Or you can load that model into the OpenAI Playground (in the UI) to validate it from there.
from openai import OpenAI
client = OpenAI()

completion = client.responses.create(
  model=fine_tuned_model_id,
  input=[
    {"role": "system", "content": "You are Elle, a factual chatbot that answers questions about elements in the periodic table with a limerick"},
    {"role": "user", "content": "Tell me about Strontium"},
  ],
  store=False,
)
print(completion.output_text)


ChatCompletionMessage(content="Strontium, a metal so bright - It's in fireworks, a dazzling sight - It's in bones, you see - And in tea, it's the key - It's the fortieth, so pure, that's the right", role='assistant', function_call=None, tool_calls=None)


---

### 第3.2步：在Playground中加載並測試微調模型

您現在可以用兩種方式來測試微調後的模型。首先，您可以訪問Playground，並使用模型下拉選單從列表中選擇您新微調的模型。另一種選擇是在微調面板中使用顯示的“Playground”選項（請參見上方截圖），它會啟動以下的_比較_視圖，該視圖將基礎模型和微調模型版本並排顯示，方便快速評估。

![Fine-tuning job status](../../../../../translated_images/zh-TW/fine-tuned-playground-compare.56e06f0ad8922016.webp)

只需填寫您訓練資料中使用的系統上下文，並提供您的測試問題。您會注意到兩邊都會更新為相同的上下文和問題。運行比較，您將看到它們輸出之間的差異。_注意微調模型如何以您示例中提供的格式來呈現回應，而基礎模型則僅遵循系統提示。_

![Fine-tuning job status](../../../../../translated_images/zh-TW/fine-tuned-playground-launch.5a26495c983c6350.webp)

您會注意到比較還會提供每個模型的標記數以及推理所花費的時間。<strong>此特定範例是用來展示流程的簡單範例，並不是真實世界的資料集或情境。</strong>您可能會注意到兩個範例顯示相同數量的標記（系統上下文和使用者提示均相同），而微調模型推理所需時間較長（自訂模型）。

在真實世界情境中，您不會使用這類玩具範例，而會針對真實資料進行微調（例如：客戶服務的產品目錄），屆時回應的質量將更加明顯。在_該_情境下，要用基礎模型獲得相同的回應品質，需要更多自訂提示工程，這會增加標記使用量，並可能延長推理相關的處理時間。_您可以嘗試查看OpenAI Cookbook中的微調範例以開始進行。_

---


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
此文件已使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們努力追求準確性，但請注意自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應視為權威來源。對於關鍵資訊，建議採用專業人工翻譯。我們不對因使用此翻譯所產生的任何誤解或誤譯承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
